# 01 · NoLA teaching walkthrough: EuroSAT

## 0. Locate sources, data and weights

In [ ]:
from pathlib import Path
import sys

# Use the Colab workspace or the local checkout.
if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/NoLA-Final-Evaluation")
    root = Path("/content/nola")
    sys.path.insert(0, str(BASE))
    from setup_inputs import setup

    root, data_dir = setup(root, root / "data", ["eurosat"], source=BASE / "source.zip")
else:
    root = Path.home() / "nola-walkthrough"
    data_dir = root / "data"

upstream = root / "upstream"
sys.dont_write_bytecode = True
sys.path[:0] = [str(upstream), str(root / ".deps")]

# Small teaching run: sampling, anchor budget, and training settings.
seed = 42
train_size, eval_size = 160, 80
eval_split = "val"
anchor_cap = 4
clip_score_batch = inference_batch = 8
head_epochs, head_batch, head_lr = 3, 16, 0.001
visual_prompts, prompt_epochs, prompt_batch, prompt_lr = 16, 1, 8, 0.001
prompt_label_smoothing = 0.1


## 1. Open EuroSAT

In [ ]:
import json
import random

import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display
from PIL import Image
from clip import clip
from utils.data_utils import ds_specific_templates
from utils.model_utils import te_transform, tr_transforms

random.seed(seed)
torch.manual_seed(seed)

# Load the published split and sample the teaching subsets.
image_dir = data_dir / "eurosat/EuroSAT_RGB"
split_path = image_dir.parent / "split_zhou_EuroSAT.json"
split = json.loads(split_path.read_text())
# Keep class names in label-ID order so scores map back to the right class.
name_by_id = {label: name for _, label, name in split["train"]}
names = [name_by_id[i] for i in range(len(name_by_id))]
train = random.Random(seed).sample(split["train"], train_size)
eval_rows = random.Random(seed).sample(split[eval_split], eval_size)


def load_images(rows, transform):
    batch = []
    for path, _, _ in rows:
        with Image.open(image_dir / path) as image:
            batch.append(transform(image.convert("RGB")))
    # [batch, 3, 224, 224]
    return torch.stack(batch).cuda()


In [ ]:
def show_tile(row):
    print(row[0])
    with Image.open(image_dir / row[0]) as image:
        display(image.convert("RGB").resize((192, 192)))


show_tile(train[0])


## 2. Make class description embeddings (CDE)

In [ ]:
model, width, preprocess = clip.load(str(root / "weights/ViT-B-32.pt"), device="cuda")
model.eval()
descriptions = json.loads((upstream / "descriptions/generic/EuroSAT.json").read_text())
templates = ds_specific_templates["EuroSAT"]

# Encode descriptions and templates separately, then build one unit vector per class.
weights = []
with torch.no_grad():
    for name in names:
        texts = descriptions[name] + [t.format(name) for t in templates]
        tokens = clip.tokenize(texts).cuda()
        vectors = F.normalize(model.encode_text(tokens), dim=1)
        weights.append(F.normalize(vectors.mean(dim=0), dim=0))
# Stack class vectors as columns: [512 features, 10 classes].
cde = torch.stack(weights, dim=1)


In [ ]:
print("CDE weights [features, classes]:", tuple(cde.shape))
preview_class = train[0][1]
print("Description:", descriptions[names[preview_class]][0])
print("Template:", templates[0].format(names[preview_class]))
print("CDE column[:6]:", cde[:6, preview_class].float().cpu().tolist())


## 3. Score tiles with unprompted CLIP

In [ ]:
# Score image embeddings against all class vectors, keeping row order.
@torch.no_grad()
def clip_scores(rows):
    scores = []
    for i in range(0, len(rows), clip_score_batch):
        images = load_images(rows[i:i+clip_score_batch], preprocess)
        features = F.normalize(model.encode_image(images), dim=1)
        logits = model.logit_scale.exp() * (features @ cde)
        scores.append(logits.softmax(dim=1).cpu())
    return torch.cat(scores)


probs = clip_scores(train)


In [ ]:
preview_pixels = load_images(train[:1], preprocess)
with torch.no_grad():
    preview_clip = F.normalize(model.encode_image(preview_pixels), dim=1)
print("Pixels:", tuple(preview_pixels.shape), "| CLIP image embedding:", tuple(preview_clip.shape))
print("CLIP image embedding[:6]:", preview_clip[0, :6].float().cpu().tolist())
print("Adaptation probabilities [images, classes]:", tuple(probs.shape))
del preview_pixels, preview_clip

eval_labels = torch.tensor([label for _, label, _ in eval_rows])
clip_eval_pred = clip_scores(eval_rows).argmax(dim=1)
clip_correct = (clip_eval_pred == eval_labels).sum().item()
print(f"Frozen CLIP CDE validation: {clip_correct}/{len(eval_labels)} ({clip_correct / len(eval_labels):.1%})")


## 4. Select anchors and pick one to trace

In [ ]:
import pandas as pd

audit = torch.tensor([label for _, label, _ in train])
confidence, predicted = probs.topk(3, dim=1)

# Rank candidates per class. Try ranks 2 and 3 only when the earlier rank is empty.
selected = []
for class_id in set(audit.tolist()):
    for rank in range(3):
        candidates = torch.where(predicted[:, rank] == class_id)[0]
        if len(candidates):
            ranked = pd.Series(confidence[candidates, rank].tolist(), index=candidates.tolist())
            chosen = ranked.sort_values(ascending=False).head(anchor_cap).index.tolist()
            selected.extend((row_id, class_id) for row_id in chosen)
            break
    else:
        raise NotImplementedError(f"No top-three candidates for {names[class_id]}")

# Keep the first assignment if the same image was selected for multiple classes.
anchors = pd.DataFrame(selected, columns=["row_id", "pseudo"]).drop_duplicates("row_id")
anchor_ids = torch.tensor(anchors["row_id"].tolist())
pseudo = torch.tensor(anchors["pseudo"].tolist())


In [ ]:
trace_id = anchor_ids[0].item()
trace_row = train[trace_id]
show_tile(trace_row)
values, ids = probs[trace_id].topk(3)
print("CLIP top three:", [(names[i], round(v, 3)) for i, v in zip(ids.tolist(), values.tolist())])
trace_target = pseudo[0].item()
print("Anchor target:", names[trace_target], "| audit class:", trace_row[2])
print("A description of the anchor target:", descriptions[names[trace_target]][0])
print("Anchor target CDE column[:6]:", cde[:6, trace_target].float().cpu().tolist())
with torch.no_grad():
    trace_clip = F.normalize(model.encode_image(load_images([trace_row], preprocess)), dim=1)
print("Traced anchor CLIP image embedding[:6]:", trace_clip[0, :6].float().cpu().tolist())
del trace_clip
print("Anchor row indices:", tuple(anchor_ids.shape), "| pseudo-labels:", tuple(pseudo.shape))

anchor_correct = (pseudo == audit[anchor_ids]).sum().item()
print(f"Anchor pseudo-label precision: {anchor_correct}/{len(anchor_ids)} ({anchor_correct / len(anchor_ids):.1%})")


## 5. Extract frozen DINO features

In [ ]:
# Free GPU space before loading the frozen DINO encoder.
model.cpu()
torch.cuda.empty_cache()
# Both checkouts have a module called utils. Temporarily allow the local DINO import to find its own.
nola_utils = sys.modules.pop("utils")
try:
    dino = torch.hub.load(str(root / "dino-source"), "dino_vitb16", source="local", pretrained=False)
finally:
    sys.modules["utils"] = nola_utils
state = torch.load(root / "weights/dino_vitbase16_pretrain.pth", map_location="cpu", weights_only=True)
dino.load_state_dict(state)
del state
dino.eval().requires_grad_(False).cuda()


# Cache one DINO feature row per training image.
@torch.no_grad()
def dino_features(rows):
    return torch.cat([dino(load_images(rows[i:i+inference_batch], te_transform)).cpu()
                      for i in range(0, len(rows), inference_batch)])


train_features = dino_features(train)


In [ ]:
print("Frozen DINO features:", tuple(train_features.shape))
print("Traced anchor DINO features[:6]:", train_features[trace_id, :6].tolist())

eval_features = dino_features(eval_rows)


## 6. Fit the DINO classifier head

In [ ]:
# Fit only the alignment head on the selected anchors.
dino_head = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, len(names)))
head_optimizer = torch.optim.Adam(dino_head.parameters(), lr=head_lr)
for epoch in range(head_epochs):
    for ids in torch.randperm(len(anchor_ids)).split(head_batch):
        logits = dino_head(train_features[anchor_ids[ids]])
        loss = F.cross_entropy(logits, pseudo[ids])
        head_optimizer.zero_grad()
        loss.backward()
        head_optimizer.step()

# Use the fitted head to assign hard targets to the full training subset.
with torch.no_grad():
    dino_train_logits = dino_head(train_features)
    dino_pseudo_labels = dino_train_logits.argmax(dim=1)


In [ ]:
print("DINO head logits [images, classes]:", tuple(dino_train_logits.shape))
print("Traced anchor head logits[:6]:", dino_train_logits[trace_id, :6].tolist())
print("Traced anchor CLIP target:", names[pseudo[0].item()],
      "| DINO target for prompting:", names[dino_pseudo_labels[trace_id].item()])

with torch.no_grad():
    dino_eval_pred = dino_head(eval_features).argmax(dim=1)
dino_correct = (dino_eval_pred == eval_labels).sum().item()
print(f"DINO alignment head validation: {dino_correct}/{len(eval_labels)} ({dino_correct / len(eval_labels):.1%})")


## 7. Insert visual prompts into CLIP

In [ ]:
dino.cpu()
torch.cuda.empty_cache()
model.float().cuda().eval()
# Freeze the backbone except visual normalization parameters, following the released name filter.
for name, param in model.named_parameters():
    param.requires_grad_("visual" in name and ("ln" in name or "bn" in name))

# Initialize shared visual tokens and a classifier seeded with the CDE.
hidden = model.visual.class_embedding.numel()
bound = (6 / (3 * 16 * 16 + hidden)) ** 0.5
prompt = nn.Parameter(torch.empty(1, visual_prompts, hidden, device="cuda").uniform_(-bound, bound))
classifier = nn.Linear(width, len(names), bias=False).cuda()
with torch.no_grad():
    classifier.weight.copy_(cde.T)


def prompted_clip_logits(images):
    tokens = model.visual.embeddings_patch(images)
    # Expand the shared prompt across the batch and insert it between CLS and the image patches.
    tokens = torch.cat([tokens[:, :1], prompt.expand(len(images), -1, -1), tokens[:, 1:]], dim=1)
    return classifier(model.visual.forward_after_patch_embeddings(tokens))


In [ ]:
with torch.no_grad():
    tokens = model.visual.embeddings_patch(load_images([trace_row], preprocess))
    logits = prompted_clip_logits(load_images([trace_row], preprocess))
print("Image tokens [batch, CLS + patches, hidden]:", tuple(tokens.shape))
print("Prompt tokens [1, prompts, hidden]:", tuple(prompt.shape))
print("First prompt token[:6] before tuning:", prompt[0, 0, :6].detach().cpu().tolist())
print("Traced anchor classifier logits before tuning[:6]:", logits[0, :6].cpu().tolist())

# Use the same validation predictor before and after tuning.
@torch.no_grad()
def prompted_predict(rows):
    return torch.cat([
        prompted_clip_logits(load_images(rows[i:i + inference_batch], te_transform)).argmax(dim=1).cpu()
        for i in range(0, len(rows), inference_batch)
    ])


prompt_before_pred = prompted_predict(eval_rows)
before_correct = (prompt_before_pred == eval_labels).sum().item()
print(f"CLIP with initial prompts: {before_correct}/{len(eval_labels)} ({before_correct / len(eval_labels):.1%})")


## 8. Tune the prompts, then compare one held-out prediction

![Weak and two strong views of the same EuroSAT tiles](attachment:strong-views.png)

In [ ]:
from PIL import ImageDraw

# Undo CLIP normalization only for display.
mean = torch.tensor([0.48145466, 0.4578275, 0.40821073])[:, None, None]
std = torch.tensor([0.26862954, 0.26130258, 0.27577711])[:, None, None]


def display_view(tensor):
    rgb = (tensor * std + mean).clamp(0, 1)
    return Image.fromarray((rgb.permute(1, 2, 0) * 255).round().byte().numpy())


size = 224
preview = Image.new("RGB", (3 * size, 3 * (size + 22) + 24), "white")
labels = ImageDraw.Draw(preview)
for col, title in enumerate(("Weak view", "Strong view 1", "Strong view 2")):
    labels.text((col * size + 5, 5), title, fill="black")

# Keep preview randomness separate from training.
state = random.getstate()
with torch.random.fork_rng(devices=[]):
    random.seed(7)
    torch.random.default_generator.manual_seed(7)
    for row, (path, _, name) in enumerate(train[:3]):
        with Image.open(image_dir / path) as file:
            image = file.convert("RGB")
        weak, strong_one = tr_transforms(image)
        _, strong_two = tr_transforms(image)
        y = 24 + row * (size + 22)
        labels.text((5, y + 3), name, fill="black")
        for col, pixels in enumerate((weak, strong_one, strong_two)):
            preview.paste(display_view(pixels), (col * size, y + 22))
random.setstate(state)
display(preview)


In [ ]:
def strong(image):
    return tr_transforms(image)[1]


# Optimize the prompts, classifier, and enabled visual normalization weights.
trainable = [(name, p) for name, p in model.named_parameters() if p.requires_grad]
prompt_optimizer = torch.optim.AdamW([
    {"params": [prompt, *classifier.parameters(), *(p for n, p in trainable if "bias" not in n)]},
    {"params": [p for n, p in trainable if "bias" in n], "weight_decay": 0.0},
], lr=prompt_lr)

# Tune CLIP against fixed DINO targets with smoothed cross-entropy.
for epoch in range(prompt_epochs):
    for ids in torch.randperm(len(train)).split(prompt_batch):
        images = load_images([train[i] for i in ids.tolist()], strong)
        # Pair the strong view with the DINO target from the weak view.
        targets = dino_pseudo_labels[ids].cuda()
        prompt_optimizer.zero_grad()
        loss = F.cross_entropy(prompted_clip_logits(images), targets, label_smoothing=prompt_label_smoothing)
        loss.backward()
        prompt_optimizer.step()


In [ ]:
print("First prompt token[:6] after tuning:", prompt[0, 0, :6].detach().cpu().tolist())
with torch.no_grad():
    logits = prompted_clip_logits(load_images([trace_row], preprocess))
print("Traced anchor classifier logits after tuning[:6]:", logits[0, :6].cpu().tolist())

prompt_final_pred = prompted_predict(eval_rows)
final_correct = (prompt_final_pred == eval_labels).sum().item()
print(f"CLIP after prompt tuning: {final_correct}/{len(eval_labels)} ({final_correct / len(eval_labels):.1%})")

# Compare all stages on the same held-out tile.
predictions = {
    "clip_cde": clip_eval_pred,
    "dino_head": dino_eval_pred,
    "clip_prompt_initial": prompt_before_pred,
    "clip_prompt_final": prompt_final_pred,
}
show_tile(eval_rows[0])
print("Held-out tile reference:", eval_rows[0][2])
for stage, pred in predictions.items():
    print(stage, "→", names[pred[0].item()])
